# Softmax Function - Practice Lab (Custom Dataset Version)

This notebook is my own version of the Softmax Function lab from Course 2, Week 2 of the Machine Learning Specialization. Instead of the original 4-blob dataset, I built this using a **5-class synthetic "fruit sensor" dataset** to practice the same concepts:

- How the softmax function converts raw scores (logits) into probabilities
- How softmax is used as an output activation in a neural network for multiclass classification
- The difference between the "obvious" (softmax-in-last-layer) and "preferred" (softmax-in-loss, `from_logits=True`) implementations in TensorFlow
- Why the preferred method is more numerically stable


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.datasets import make_blobs
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)
np.set_printoptions(precision=3, suppress=True)

> **Note on indexing:** Like the original lab, most equations below are written with 1-to-N indexing (matching lecture notation), while the code uses 0-to-N-1 indexing (matching Python/NumPy).

## 1. The Softmax Function

Given a vector of raw scores (logits) $\mathbf{z} = [z_1, z_2, \dots, z_N]$, softmax converts it into a probability distribution:

$$a_j = \frac{e^{z_j}}{\sum_{k=1}^{N} e^{z_k}} \tag{1}$$

Each $a_j$ is between 0 and 1, and all $a_j$ sum to 1, so they can be read as probabilities. Larger inputs get relatively larger output probabilities because of the exponential.

Let's implement it in NumPy.

In [ ]:
def my_softmax(z):
    ez = np.exp(z)              # element-wise exponential
    sm = ez / np.sum(ez)
    return sm

# quick sanity check
z_test = np.array([1.0, 2.0, 3.0, 4.0])
a_test = my_softmax(z_test)
print("z:", z_test)
print("softmax(z):", a_test)
print("sum of outputs:", np.sum(a_test))

### Visualizing softmax

Instead of the original lab's interactive slider widget, let's visualize softmax by plotting several different `z` vectors side-by-side as bar charts, so we can see how the shape of `z` changes the output probabilities.

In [ ]:
z_examples = {
    "Balanced":        np.array([1.0, 1.0, 1.0, 1.0, 1.0]),
    "Slightly skewed": np.array([1.0, 2.0, 1.5, 1.0, 0.5]),
    "One dominant":    np.array([1.0, 1.0, 5.0, 1.0, 1.0]),
    "Wide spread":     np.array([-2.0, 0.0, 2.0, 4.0, 6.0]),
}

fig, axes = plt.subplots(1, len(z_examples), figsize=(16, 4))
labels = ["c1", "c2", "c3", "c4", "c5"]

for ax, (title, z) in zip(axes, z_examples.items()):
    a = my_softmax(z)
    ax.bar(labels, a, color="teal")
    ax.set_ylim(0, 1)
    ax.set_title(f"{title}\nz={z}")
    ax.set_ylabel("probability")

plt.tight_layout()
plt.show()

Things to notice:
- The exponential magnifies differences: a small gap in `z` can become a large gap in probability (see "One dominant").
- The outputs always sum to 1.
- Softmax is a joint function of *all* the inputs — changing one `z` value shifts every output probability, unlike ReLU or Sigmoid which act independently on each input.

## 2. Cost Function

The loss associated with softmax is the **cross-entropy loss**:

$$
L(\mathbf{a}, y) =
\begin{cases}
-\log(a_1), & \text{if } y = 1 \\
\quad\vdots \\
-\log(a_N), & \text{if } y = N
\end{cases} \tag{2}
$$

Using the indicator function $\mathbf{1}\{y == n\}$ (1 if true, 0 otherwise), the cost over $m$ examples with $N$ classes is:

$$
J(\mathbf{w}, b) = -\frac{1}{m} \left[ \sum_{i=1}^{m} \sum_{j=1}^{N} \mathbf{1}\{y^{(i)} == j\} \log \frac{e^{z_j^{(i)}}}{\sum_{k=1}^{N} e^{z_k^{(i)}}} \right] \tag{3}
$$

Only the term matching the true label $y^{(i)}$ contributes to each example's loss.

## 3. Building a Custom Dataset

The original lab used 4 Gaussian blobs. Here I'll create a **5-class dataset** instead — think of it as 5 different "sensor score" clusters (e.g. 5 categories of fruit based on two measured features like sweetness and firmness). This still lets us practice the exact same multiclass classification workflow, just with different data.

In [ ]:
# Custom 5-class synthetic dataset
centers = [[-6, 3], [-3, -3], [0, 4], [3, -2], [6, 3]]
X_train, y_train = make_blobs(
    n_samples=2500,
    centers=centers,
    cluster_std=1.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("classes:", np.unique(y_train))

In [ ]:
plt.figure(figsize=(6, 6))
colors = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db", "#9b59b6"]
for c in range(5):
    mask = y_train == c
    plt.scatter(X_train[mask, 0], X_train[mask, 1], s=10, color=colors[c], label=f"class {c}")
plt.legend()
plt.title("Custom 5-class synthetic dataset")
plt.xlabel("feature 1")
plt.ylabel("feature 2")
plt.show()

## 4. The "Obvious" Organization

Softmax is applied directly as the activation of the final Dense layer. The loss function (`SparseCategoricalCrossentropy`) then receives probabilities.

In [ ]:
model = Sequential([
    Dense(25, activation='relu'),
    Dense(15, activation='relu'),
    Dense(5, activation='softmax')   # < softmax activation here, 5 classes now
])

model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

model.fit(X_train, y_train, epochs=10)

In [ ]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred[:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

## 5. The Preferred Organization

Here the final layer uses a **linear** activation (i.e. no activation — raw logits), and the loss function does the softmax internally via `from_logits=True`. This is more numerically stable, especially for very confident predictions where `exp()` can overflow/underflow.

In [ ]:
preferred_model = Sequential([
    Dense(25, activation='relu'),
    Dense(15, activation='relu'),
    Dense(5, activation='linear')   # <-- linear, not softmax
])

preferred_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  # <-- Note
    optimizer=tf.keras.optimizers.Adam(0.001),
)

preferred_model.fit(X_train, y_train, epochs=10)

In [ ]:
p_preferred = preferred_model.predict(X_train)
print(f"two example output vectors (logits):\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

Notice the preferred model's raw outputs are **not** probabilities — they can be any real number (positive or negative). To turn them into probabilities, we pass them through softmax ourselves.

In [ ]:
sm_preferred = tf.nn.softmax(p_preferred).numpy()
print(f"two example output vectors (probabilities):\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

If you only need the **predicted class** (not the probability), you don't even need softmax — `np.argmax` on the raw logits gives the same answer, since softmax is monotonic.

In [ ]:
for i in range(5):
    print(f"logits: {p_preferred[i]}, predicted class: {np.argmax(p_preferred[i])}, true class: {y_train[i]}")

In [ ]:
# Quick accuracy check on the training set using the preferred model
predicted_classes = np.argmax(p_preferred, axis=1)
accuracy = np.mean(predicted_classes == y_train)
print(f"Training accuracy: {accuracy*100:.2f}%")

## 6. SparseCategoricalCrossentropy vs CategoricalCrossentropy

TensorFlow offers two related loss functions depending on how your labels are formatted:

- **SparseCategoricalCrossentropy**: labels are plain integers (e.g. `y = 2` for the 3rd class out of 5). This is what we used above (`y_train` is an array of ints 0-4).
- **CategoricalCrossentropy**: labels are one-hot encoded vectors (e.g. class 2 out of 5 → `[0, 0, 1, 0, 0]`).

Both compute the same underlying cross-entropy loss — the difference is purely about label format.

## Summary

In this notebook I practiced:
- Implementing the softmax function from scratch in NumPy
- Visualizing how softmax turns logits into a probability distribution
- Building a 5-class synthetic dataset (different from the course's original 4-class one)
- The "obvious" method: softmax activation in the last Dense layer
- The "preferred" method: linear last layer + `from_logits=True`, which is more numerically stable
- Recovering probabilities from logits with `tf.nn.softmax`, and recovering the predicted class with `np.argmax`

This mirrors the original Course 2 Week 2 Softmax lab from the Machine Learning Specialization, but built end-to-end with new data so I could practice the workflow myself rather than just running the provided notebook.